# Exercise 6: Checkpoint + State + Recover

This exercise will be a full implementation of a Stream with Auto Loader (Compatible with Databircks Free Edition).

Then an Aggregation will be performed with State (by id).

Will create a Checkpoint in Volumes.

Since we are in free edition the trigger we will make is having `availableNow=True`.

This pipeline will be executed many times to prove that the state remains without duplicity.

First we begin by setting up the volume, folders and output Delta Table

In [0]:
%sql
-- Step 0: Create the Volumen if it does not exist
CREATE VOLUME IF NOT EXISTS workspace.default.streaming_demo;

In [0]:
# Step 0.1: Configure either input state and checkpoint state folders
dbutils.fs.mkdirs("/Volumes/workspace/default/streaming_demo/input_state")
dbutils.fs.mkdirs("/Volumes/workspace/default/streaming_demo/chk_state")

True

In [0]:
%sql
-- Step 0.2: Creation of the Delta Table in the volume
CREATE TABLE IF NOT EXISTS workspace.default.streaming_state_demo (
  id BIGINT,
  total_events BIGINT
);

Now we are going to create an input Stream with Auto Loader and timestamp as a column

In [0]:
# Step 1: Stream Creation
from pyspark.sql import functions as F

df_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/streaming_demo/schema_state")
    .load("/Volumes/workspace/default/streaming_demo/input_state")
    .withColumn("timestamp", F.current_timestamp())
)

To test this Stream we are going to set a JSON as input via `dbutils.fs.put()` 

In [0]:
# Step 1.1: Upload a JSON file to the input_state folder
dbutils.fs.put(
    "/Volumes/workspace/default/streaming_demo/input_state/batch3.json",
    """{"id": 1}
{"id": 1}
{"id": 2}"""
)

Wrote 29 bytes.


True

Now, let's perform the aggregation with State. This aggregation will be an accumulated count by id

In [0]:
# Step 2: Create the aggregation.
## Note: This is stateful because Sparkkeeps the count by id between executions using the checkpint
agg_state = (
    df_stream
    .groupBy("id")
    .agg(F.count("*").alias("total_events"))
)

In [0]:
# Step 3: Stream writing with checkpoint + availableNow
## Note: When availableNow ends up running, it will process everything and then will stop
query = (
    agg_state.writeStream
        .format("delta")
        .outputMode("complete")
        .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/chk_state")
        .trigger(availableNow=True)
        .table("workspace.default.streaming_state_demo")
)

Then we have to see the aggregated State

In [0]:
%sql
-- Step 4: Aggregated State test
SELECT * FROM workspace.default.streaming_state_demo ORDER BY id;

id,total_events
1,5
2,2
3,1


Now we have to test the recovery, or the state mainteinance. To do this, we will simulate new events

In [0]:
# Step 5: New Event Simulation
dbutils.fs.put(
    "/Volumes/workspace/default/streaming_demo/input_state/batch4.json",
    """{"id": 4}
{"id": 3}"""
)


Wrote 19 bytes.


True

Then we go back to `Step 3`, and re-run that block.

Finally we check again using `Step 4` query

In [0]:
%sql
SELECT * FROM workspace.default.streaming_state_demo ORDER BY id;

id,total_events
1,5
2,2
3,2
4,1
